# Produce and replay formal safety certificates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/10_formal_certificates.ipynb)

For safe models, TensorGuard can attach a replayable safety certificate and an optional proof-certificate DAG to the same top-level result used by CI.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
from tensorguard import verify_architecture

source = '''
import torch.nn as nn
class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
    def forward(self, x):
        return self.fc(x)
'''
r = verify_architecture(source, input_shapes={'x': ('batch', 10)},
                        produce_certificates=True, max_cegar_iterations=0)
print('verdict:', r.verdict)
print('certificate replay:', r.certificate_replay.ok,
      r.certificate_replay.verification_conditions)
assert r.verdict == 'SAFE'
assert r.safety_certificate is not None
assert r.proof_certificate is not None
assert r.proof_certificate.verify_locally()
assert r.certificate_replay.ok

Unsafe or out-of-fragment models deliberately do not receive a safe certificate:

In [ ]:
bad = source.replace('nn.Linear(10, 5)', 'nn.Linear(9, 5)')
unsafe = verify_architecture(bad, input_shapes={'x': ('batch', 10)},
                             produce_certificates=True)
print('unsafe verdict:', unsafe.verdict)
assert unsafe.verdict == 'UNSAFE'
assert unsafe.safety_certificate is None